# QKeyGuard: BB84 with and without Eve

A small, reproducible walk-through of the circuit-based simulator. Quantum outcomes come from Qiskit Aer; NumPy supplies classical protocol choices.

In [ ]:
from src.bb84_protocol import simulate_bb84
from src.security_analysis import security_status


## Inspect one transmitted qubit

The Z basis uses computational states. Applying H before measurement selects the X basis. A matching measurement basis recovers Alice's bit in an ideal simulation.

In [ ]:
ideal = simulate_bb84(n_qubits=256, eve=False, seed=84)
print(ideal.circuit.draw(output='text'))
print(f'Sifted positions: {len(ideal.matching_indices)}')
print(f'Test QBER: {ideal.test_qber:.1%}' if ideal.test_qber is not None else 'No test sample')

## Add intercept-and-resend

Eve's measured bit and basis determine the replacement state sent to Bob. In the ideal protocol, the expected sifted error rate is 25%, with statistical variation at finite signal counts.

In [ ]:
attacked = simulate_bb84(n_qubits=4096, eve=True, seed=84)
print(f'Sifted positions: {len(attacked.matching_indices)}')
print(f'Test-sample QBER: {attacked.test_qber:.1%}' if attacked.test_qber is not None else 'No test sample')
print(f'Diagnostic full sifted QBER: {attacked.sifted_qber:.1%}' if attacked.sifted_qber is not None else 'No sifted bits')
print(security_status(attacked))
attacked.circuit.draw(output='mpl')

## Key accounting

The disclosed test bits are excluded from the remaining key candidates. A real protocol must then reconcile errors and apply privacy amplification; the candidates shown here are not certified secret key material.

In [ ]:
print('Sifted raw-key length:', len(attacked.matching_indices))
print('Disclosed sample length:', len(attacked.test_indices))
print('Remaining candidate length:', len(attacked.secret_indices))